# 11c NOD development analysis on the nine donors

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 11a and 11b.

Development analysis, not a prespecified test. Donors only; the 21 target people are not read. Each donor plays a new person with its first session (1,000
images): 200 held out as test, k = 250 and 500 labelled from the rest, borrowing from the other eight donors' ridge
encoders. Vertices: mean cross-validated r of the other donors' own models above 0.05 (primary) or 0.1. The outputs
set P-8's margins.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '11c-noddryrun', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"donors": ["sub-01", "sub-02", "sub-03", "sub-04", "sub-05", "sub-06", "sub-07", "sub-08", "sub-09"], "K": 200, "ks": [250, 500], "draws": [13, 14, 15], "folds": 5, "n_test": 200, "session_trials": 1000, "test_seed": 374, "cv_eval_share": 0.2, "cv_seed": 7, "thresholds": [0.05, 0.1], "primary_threshold": 0.05, "features": ["alexnet", "dinov2"], "wall_hours": 11.0, "encoder": "ridge", "min_vertices": 100}')

import numpy as np
import pandas as pd
from cross_subject_encoding.eval import lowrank_map as lm
from cross_subject_encoding.eval import nsd_confirm as nc
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C

IN = Path("/kaggle/input")
nod = next(p.parent for p in IN.rglob("sub-01_images.json"))
Y, IMG = {}, {}
for s in FIXED["donors"]:
    Y[s] = np.load(nod / f"{s}_betas.npy", mmap_mode="r")
    IMG[s] = np.array([n.replace(".JPEG", "") for n in json.loads((nod / f"{s}_images.json").read_text())])
V = Y[FIXED["donors"][0]].shape[1]
FEATS = {}
for name in FIXED["features"]:
    f = np.load(next(IN.rglob(f"11b_{name}_kpca1000.npz")))
    FEATS[name] = (f["z"].astype(np.float64), {str(n): j for j, n in enumerate(f["names"])})
feats = lambda name, names: FEATS[name][0][[FEATS[name][1][str(n)] for n in names]]
print({s: Y[s].shape for s in FIXED["donors"]}, "|", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
def ols_coef(x, y):
    # ridge with a per-vertex penalty (5-fold CV on these images), returned as [intercept; coefficients] like OLS
    m = RidgePerOutput(lm.ALPHAS, FIXED["folds"], 13).fit(x, np.asarray(y, np.float32))
    print("   alpha median", float(np.median(m.best_alpha_)), "at grid edge", round(m.alpha_at_edge_, 3), flush=True)
    return np.vstack([(m.y_mean_ - m.x_mean_ @ m.coef_)[None], m.coef_]).astype(np.float32)
ols_pred = lambda beta, x: (np.concatenate([np.ones((len(x), 1)), x], 1) @ beta.astype(np.float64))
def r_cols(p, m):
    p, m = p - p.mean(0), m - m.mean(0)
    return (p * m).sum(0) / np.sqrt((p ** 2).sum(0) * (m ** 2).sum(0) + 1e-30)
BETA, RMAP = {}, {}
for s in FIXED["donors"]:
    y = np.asarray(Y[s], np.float32)
    for name in FIXED["features"]:
        BETA[(name, s)] = ols_coef(feats(name, IMG[s]), y)
    # vertex reliability for selection: the donor's own AlexNet model, fitted on 80% of its images, scored on 20%
    order = np.random.default_rng(FIXED["cv_seed"]).permutation(len(y))
    n_ev = int(FIXED["cv_eval_share"] * len(y))
    ev, fi = order[:n_ev], order[n_ev:]
    b = ols_coef(feats("alexnet", IMG[s][fi]), y[fi])
    RMAP[s] = r_cols(ols_pred(b, feats("alexnet", IMG[s][ev])), y[ev]).astype(np.float32)
    del y
    print(s, "encoders; cv r median / 99th pct / >0.05 / >0.1:", round(float(np.median(RMAP[s])), 4),
          round(float(np.percentile(RMAP[s], 99)), 4), int((RMAP[s] > 0.05).sum()), int((RMAP[s] > 0.1).sum()), "|", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)

In [ ]:
def oof(fit_predict, k, seed):
    split = np.array_split(np.random.default_rng(seed).permutation(k), FIXED["folds"])
    out = None
    for held in split:
        keep = np.setdiff1d(np.arange(k), held)
        pred = fit_predict(keep, held)
        if out is None:
            out = np.empty((k, pred.shape[1]))
        out[held] = pred
    return out

rows, sel_rows = [], []
for p, s in enumerate(FIXED["donors"], start=1):
    t = time.monotonic()
    donors = [d for d in FIXED["donors"] if d != s]
    session1 = np.arange(FIXED["session_trials"])  # the first ImageNet session, in trial order
    order = np.random.default_rng([FIXED["test_seed"], p]).permutation(session1)
    test, pool = np.sort(order[:FIXED["n_test"]]), np.sort(order[FIXED["n_test"]:])
    mean_r = np.mean([RMAP[d] for d in donors], 0)
    masks = {thr: np.where(mean_r > thr)[0] for thr in FIXED["thresholds"]}
    for thr, m in masks.items():
        sel_rows.append({"subject": s, "threshold": thr, "vertices": int(len(m))})
    y_all = np.asarray(Y[s], np.float32)
    X = {f: (feats(f, IMG[s][pool]), feats(f, IMG[s][test])) for f in FIXED["features"]}
    for thr, cols in masks.items():
        if len(cols) < FIXED["min_vertices"]:
            print(f"  {s}: {len(cols)} vertices at {thr}, below {FIXED['min_vertices']}: skipped")
            continue
        meas = y_all[test][:, cols].astype(np.float64)
        y_pool = y_all[pool][:, cols]
        def score(method, k, d, pred, centred=False):
            rows.append({"subject": s, "threshold": thr, "method": method, "k": k, "draw": d, "centred": centred,
                         "enc_r": nc.encoding_r(pred, meas), "top1": nc.identify_top1(pred, meas, centred=centred)})
        avg = {}
        for f in FIXED["features"]:
            acc_p, acc_t = np.zeros((len(pool), len(cols))), np.zeros((len(test), len(cols)))
            for d in donors:
                acc_p += ols_pred(BETA[(f, d)][:, cols], X[f][0])
                acc_t += ols_pred(BETA[(f, d)][:, cols], X[f][1])
            avg[f] = (acc_p / len(donors), acc_t / len(donors))
        basis = {f: lm.prediction_basis(avg[f][0], FIXED["K"]) for f in FIXED["features"]}
        score("avg", 0, "-", avg["alexnet"][1])
        score("avg", 0, "-", avg["alexnet"][1], centred=True)
        # own model on all pool images: ridge, since 800 images < 1,001 OLS parameters
        score("own_full", "all", "-", RidgePerOutput(lm.ALPHAS, FIXED["folds"], 13).fit(X["alexnet"][0], y_pool.astype(np.float64)).predict(X["alexnet"][1]))
        for d in FIXED["draws"]:
            o = np.random.default_rng([d, p]).permutation(len(pool))
            for k in FIXED["ks"]:
                idx = o[:k]
                y_k = y_pool[idx].astype(np.float64)
                parts = {}
                for f in FIXED["features"]:
                    a_k = avg[f][0][idx]
                    mean, B = basis[f]
                    m_test = lm.apply_lowrank_map(lm.fit_lowrank_map(a_k, y_k, mean, B, seed=d), avg[f][1], mean, B)
                    m_oof = oof(lambda tr, he: lm.apply_lowrank_map(lm.fit_lowrank_map(a_k[tr], y_k[tr], mean, B, seed=d), a_k[he], mean, B), k, d)
                    parts[("map", f)] = (m_oof, m_test)
                xk = X["alexnet"][0][idx]
                own = RidgePerOutput(lm.ALPHAS, FIXED["folds"], d).fit(xk, y_k)
                parts["own"] = (oof(lambda tr, he: own.refit(xk[tr], y_k[tr]).predict(xk[he]), k, d), own.predict(X["alexnet"][1]))
                score("avg_labelled_mean", k, d, nc.with_labelled_mean(avg["alexnet"][1], avg["alexnet"][0][idx], y_k))
                score("lowrank_map", k, d, parts[("map", "alexnet")][1])
                score("own_ridge", k, d, parts["own"][1])
                for name, chosen in (("stack", [("map", "alexnet"), "own"]), ("mixed", [("map", "dinov2"), "own"])):
                    w = nc.stack_fit([parts[c][0] for c in chosen], y_k)
                    score(name, k, d, nc.stack_apply(w, [parts[c][0] for c in chosen], [parts[c][1] for c in chosen], y_k.mean(0)))
    del y_all
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / "11c_scores.csv", index=False)
    print(s, f"{time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min", flush=True)
    if (time.monotonic() - T0) / 3600 > FIXED["wall_hours"]:
        print("STOPPED: wall clock after", s)
        break
pd.DataFrame(sel_rows).to_csv(C.METRICS_DIR / "11c_vertices.csv", index=False)

In [ ]:
frame = pd.DataFrame(rows)
def per_subject(metric, thr, method, centred=False, ks=None):
    f = frame[(frame.threshold == thr) & (frame.method == method) & (frame.centred == centred)]
    if ks is not None:
        f = f[f.k.astype(str).isin([str(k) for k in ks])]
    return f.groupby("subject")[metric].mean()
def contrast(metric, thr, a, b):
    # as P-7: borrowing at k = 0 is scored centred in decoding (transductive, stated), raw in encoding (r is unaffected)
    ref = per_subject(metric, thr, b, centred=(b == "avg" and metric == "top1"), ks=None if b == "avg" else FIXED["ks"])
    x = (per_subject(metric, thr, a, ks=FIXED["ks"]) - ref).dropna()
    return {"mean": float(x.mean()), "sd": float(x.std(ddof=1)), "n": int(len(x)), "positive": int((x > 0).sum()),
            "per_person": {k: round(float(v), 4) for k, v in x.items()}}
summary = {"post_hoc": "development analysis on NOD donors", "levels": {}, "contrasts": {},
           "vertices": pd.DataFrame(sel_rows).groupby("threshold").vertices.median().to_dict()}
for thr in FIXED["thresholds"]:
    lv = {}
    for method in ("avg", "avg_labelled_mean", "lowrank_map", "own_ridge", "stack", "mixed", "own_full"):
        for k in ([None] if method in ("avg", "own_full") else FIXED["ks"]):
            key = method if k is None else f"{method}_k{k}"
            lv[key] = {m: float(per_subject(m, thr, method, ks=None if k is None else [k]).mean()) for m in ("enc_r", "top1")}
    lv["avg_centred"] = {m: float(per_subject(m, thr, "avg", centred=True).mean()) for m in ("enc_r", "top1")}
    summary["levels"][str(thr)] = lv
    summary["contrasts"][str(thr)] = {
        "C1_map_minus_avg_enc_r": contrast("enc_r", thr, "lowrank_map", "avg"),
        "C2_stack_minus_avg_top1": contrast("top1", thr, "stack", "avg"),
        "C3_stack_minus_own_top1": contrast("top1", thr, "stack", "own_ridge"),
        "mixed_minus_avg_top1": contrast("top1", thr, "mixed", "avg"),
        "mixed_minus_stack_top1": contrast("top1", thr, "mixed", "stack")}
(C.METRICS_DIR / "11c_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("11C_SUMMARY_JSON " + json.dumps(summary, default=str))